# <font size=6><b>Lec05. [실습] 파일유사도

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings(action='ignore')

# ----------------------------------------------------------- 토큰화
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
# ----------------------------------------------------------- 유사도
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


#-------------------- 차트 관련 속성 (한글처리, 그리드) -----------
plt.rcParams['font.family']= 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

#-------------------- 주피터 , 출력결과 넓이 늘리기 ---------------
# from IPython.core.display import display, HTML
from IPython.display import display, HTML
display(HTML("<style>.container{width:100% !important;}</style>"))
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('max_colwidth', None)

In [2]:
import os

folder = r"C:\IT\workspace_ptyhon\dl\LLM\lec05_data"

flist = []
for fname in os.listdir(folder):
    if fname.endswith(".txt"):
        path_filename = os.path.join(folder, fname)
        with open(path_filename, "r", encoding="utf-8") as f:
            #print(fname, f.read()[:80]) #한번만 읽어야햄 
            flist.append([fname, f.read()])

np.array(flist).shape

(8, 2)

In [3]:
df = pd.DataFrame(flist, columns=["fname", "content"])
df

,fname,content
0,a.txt,"유사도(Similarity)는 두 데이터 객체(문서, 벡터 등)가 얼마나 비슷한지 정량적으로 수치화한 지표로, 주로 0에서 1 사이(또는 -1~1) 값으로 표현됩니다. 높을수록(1에 가까울수록) 두 데이터가 유사하며, KCI 문헌 유사도 검사 서비스나 꼬맨틀 등에서 데이터 분석, 표절 검사, 추천 시스템의 핵심 기술로 사용됩니다. \n\n1. 주요 유사도 측정 방식\n코사인 유사도 (Cosine Similarity): 두 벡터 간의 사잇각을 이용하여 유사도를 측정하는 방식입니다. 방향의 유사성을 중요하게 보며, 각도가 좁을수록(0도에 가까울수록) 유사도가 1에 가까워집니다.\n유클리드 거리 (Euclidean Distance): 두 점 사이의 기하학적 거리를 계산하여 유사도를 측정합니다. 거리가 작을수록 유사도가 높다고 판단하며, 비유사도를 측정하는 방식입니다."
1,b.txt,"데이터 과학/AI: 군집화(Clustering), 분류(Classification), 데이터 라벨링 및 AI 모델 학습/진단에 활용됩니다.\n문서 및 코드 유사도: KCI(한국학술지인용색인) 등을 활용해 문서 간 유사성을 측정하여 표절을 검사하거나, 파이썬 등 코드 도용을 방지하기 위한 도구로 활용됩니다 KCI 문헌 유사도 검사 서비스.\n비유사도(Dissimilarity): 데이터 간의 거리를 나타내며, 0에 가까울수록 유사도가 높고 비유사도가 낮다는 것을 의미합니다. \n"
2,c.txt,전쟁은 국가 또는 정치 집단 사이의 폭력이나 무력을 사용하는 상태 또는 행동을 말한다. 특별히 둘 이상의 국가 간에 어떤 목적을 두고 수행되는 싸움이다. 그러나 독일 농민전쟁 같은 내전도 전쟁이다. 치열한 경쟁이나 혼란을 전쟁에 비유하여 말하기도 한다.
3,d.txt,전쟁은 국가 또는 정치 집단 사이의 폭력이나 무력을 사용하는 상태 또는 행동을 말한다. 특별히 둘 이상의 국가 간에 어떤 목적을 두고 수행되는 싸움이다. 그러나 독일 농민전쟁 같은 내전도 전쟁이다. 치열한 경쟁이나 혼란을 전쟁에 비유하여 말하기도 한다.
4,e.txt,"머신 러닝(Machine Learning, 기계 학습)은 컴퓨터가 명시적으로 프로그래밍되지 않고도 데이터와 경험을 통해 스스로 패턴을 학습하고 예측 모델을 빌드하여 성능을 개선하는 인공지능(AI)의 하위 분야입니다. 인공지능이 인간의 지능을 모방한다면, 머신 러닝은 데이터를 기반으로 규칙을 스스로 찾아내 의사결정을 내리는 수단입니다. \n핵심 정의 및 특징\n스스로 학습: 데이터를 분석하여 패턴을 찾고, 학습량이 많아질수록 더 빠르고 정확해집니다.\n기반 기술: 데이터와 알고리즘을 활용하여 인간과 유사한 방식으로 문제를 해결합니다.\n데이터 주도: 사람이 규칙을 코딩하는 대신, 기계가 데이터를 통해 직접 규칙을 찾습니다."
5,f.txt,"머신 러닝의 주요 유형\n지도 학습 (Supervised Learning): 라벨링된(정답이 있는) 데이터를 통해 학습하여 입력값에 대한 예측값(분류/회귀)을 학습합니다.\n비지도 학습 (Unsupervised Learning): 라벨링되지 않은 데이터에서 숨겨진 구조나 패턴을 찾아냅니다.\n강화 학습 (Reinforcement Learning): 행동에 대한 보상을 통해 스스로 최적의 행동 방식을 학습합니다. \n\n머신 러닝 활용 예시\n추천 시스템: 유튜브, 넷플릭스 등의 맞춤 콘텐츠 추천.\n이미지 및 음성 인식: 사진 속 사물 분류, 음성 인식 서비스.\n사기 탐지 및 보안: 금융 거래의 이상 징후 감지.\n언어 모델: 챗봇 및 자동 번역 서비스.\n예측 모델: 주식 트레이딩 및 에너지 부하 예측. \n\n머신 러닝의 동의어 및 관련 용어\n기계 학습\nML (Machine Learning)\n데이터 마이닝 (부분집합)\n예측 분석 (Predictive Analytics) \n머신 러닝은 일반 프로그램이 고정된 규칙대로 움직이는 것과 달리, 데이터를 통해 경험을 쌓아 업무 수행 능력을 점진적으로 향상시키는 기술입니다."
6,g.txt,"머신러닝(기계학습)은 데이터에서 규칙을 찾아 학습하는 인공지능의 한 분야이며, 딥러닝은 이를 발전시킨 인공신경망 기반의 하위 개념입니다. 가장 큰 차이는 사람의 개입 여부로, 머신러닝은 사람이 특징을 정의해줘야 하지만, 딥러닝은 인공지능이 스스로 데이터의 특징을 추출하여 학습합니다. \n\n머신러닝 vs 딥러닝 주요 차이점\n구조: 머신러닝은 다양한 알고리즘(회귀, 의사결정 나무 등)을 사용하며, 딥러닝은 인간 뇌를 모방한 다층 신경망(Deep Neural Network)을 사용합니다.\n특징 추출 (Feature Extraction): 머신러닝은 사람이 데이터의 주요 특징을 추출해 주어야 하는 반면, 딥러닝은 구조화되지 않은 데이터(이미지, 음성 등)에서 스스로 특징을 자동 추출합니다.\n데이터 양: 머신러닝은 적은 데이터로도 학습 가능하지만, 딥러닝은 고성능 GPU와 대량의 데이터가 필요합니다.\n응용 분야: 머신러닝은 금융 데이터 분석, 스팸 메일 분류에 적합하며, 딥러닝은 자율주행, 얼굴 인식, AI 번역 등 복잡한 작업에 활용됩니다."
7,h.txt,"머신 러닝 알고리즘은 데이터를 분석하여 패턴을 학습하고, 예측이나 의사결정을 수행하는 수학적 방법론입니다. 주요 유형으로 정답이 있는 데이터를 학습하는 지도 학습(회귀, 분류), 정답 없이 구조를 찾는 비지도 학습(군집화), 행동에 따른 보상을 극대화하는 강화 학습으로 나뉩니다. \n\n주요 머신 러닝 알고리즘 분류 및 종류\n지도 학습 (Supervised Learning): 입력값과 목표값의 매핑 관계를 학습합니다.\n선형 회귀 (Linear Regression): 연속적인 수치(예: 주가, 매출)를 예측.\n로지스틱 회귀 (Logistic Regression): 데이터가 특정 범주에 속할 확률을 0~1 사이로 예측하여 분류.\n의사결정 트리 (Decision Tree): 스무고개 방식으로 데이터 분류 및 회귀 수행.\n랜덤 포레스트 (Random Forest): 여러 결정 트리를 결합하여 예측 성능을 높인 앙상블 알고리즘.\n서포트 벡터 머신 (SVM): 데이터 분류 경계면(마진)을 최대화하는 최적의 선을 찾음."


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   fname    8 non-null      object
 1   content  8 non-null      object
dtypes: object(2)
memory usage: 256.0+ bytes


* 불용어 사전로드

In [5]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize 
stop_words_list = stopwords.words('my_korean')    #----C:\Users\user\..\stopwords\my_korean.txt
print('불용어 개수 :', len(stop_words_list))
print('불용어 10개 출력 :',stop_words_list[:10])

불용어 개수 : 595
불용어 10개 출력 : ['가', '가까스로', '가령', '각', '각각', '각자', '각종', '갖고말하자면', '같다', '같이']


* 명사추출

In [6]:
from konlpy.tag import Mecab
mecab = Mecab(dicpath=r"C:/mecab/mecab-ko-dic") 
print('Mecab 명사 추출 :'  , mecab.nouns ("열심히 코딩한 당신, 연휴에는 여행을 가봐요")) 

Mecab 명사 추출 : ['코딩', '당신', '연휴', '여행']


In [7]:
clean_content_list = []
for  text in df['content']:
    word_list = mecab.nouns (text)
    word_list = [w for w in word_list    if len(w) > 1]
    word_list = " ".join(word_list)
    clean_content_list.append(word_list)

clean_content_list[:2]

['유사 데이터 객체 문서 벡터 정량 수치 지표 사이 표현 데이터 유사 문헌 유사 검사 서비스 맨틀 데이터 분석 표절 검사 추천 시스템 핵심 기술 사용 주요 유사 측정 방식 코사인 유사 벡터 사잇각 이용 유사 측정 방식 방향 유사 중요 각도 유사 유클리드 거리 사이 기하학 거리 계산 유사 측정 거리 유사 판단 비유 사도 측정 방식',
 '데이터 과학 군집 분류 데이터 라벨 모델 학습 진단 활용 문서 코드 유사 한국 학술지 인용 색인 활용 문서 유사 측정 표절 검사 파이썬 코드 도용 방지 도구 활용 문헌 유사 검사 서비스 비유 사도 데이터 거리 유사 비유 사도 의미']

In [8]:
np.array(clean_content_list).shape

(8,)

In [9]:
tfidf_vt = TfidfVectorizer(binary=True)
res = tfidf_vt.fit_transform(clean_content_list)  #---------- 1D
#print(tfidf_vt.vocabulary_)

print(res.shape)
print(tfidf_vt.get_feature_names_out())

tfidf_df = pd.DataFrame(res.toarray(), columns=tfidf_vt.get_feature_names_out())
tfidf_df.head(2)

(8, 196)
['가능' '각도' '강화' '개념' '개선' '개입' '객체' '거래' '거리' '검사' '결정' '결합' '경쟁' '경험'
 '계면' '계산' '고정' '과학' '관계' '관련' '구조' '국가' '군집' '규칙' '극대' '금융' '기계' '기반'
 '기술' '기하학' '나무' '내전' '넷플릭스' '농민' '능력' '다층' '대량' '대신' '데이터' '도구' '도용' '독일'
 '동의어' '라벨' '랜덤' '러닝' '로지스틱' '마이닝' '마진' '맞춤' '매출' '매핑' '맨틀' '머신' '메일' '명시'
 '모델' '모방' '목적' '목표' '무력' '문서' '문제' '문헌' '반면' '발전' '방법론' '방식' '방지' '방향'
 '번역' '범주' '벡터' '보상' '보안' '부분집합' '부하' '분류' '분석' '분야' '비유' '비지' '사도' '사람'
 '사물' '사용' '사이' '사잇각' '사진' '상태' '색인' '서비스' '서포트' '선형' '성능' '수단' '수치' '수학'
 '수행' '스무고개' '스팸' '시스템' '신경망' '싸움' '알고리즘' '앙상블' '언어' '얼굴' '업무' '에너지' '여부'
 '연속' '예시' '예측' '용어' '유사' '유클리드' '유튜브' '유형' '음성' '응용' '의미' '의사' '이미지' '이상'
 '이용' '인간' '인공' '인공지능' '인식' '인용' '일반' '입력' '자동' '자율' '작업' '전쟁' '점진' '정답'
 '정량' '정의' '정치' '정확' '종류' '주가' '주도' '주식' '주요' '주행' '중요' '지능' '지도' '지표'
 '진단' '집단' '징후' '차이' '차이점' '최대' '최적' '추천' '추출' '측정' '치열' '컴퓨터' '코드' '코딩'
 '코사인' '콘텐츠' '탐지' '트레이딩' '트리' '특정' '특징' '파이썬' '판단' '패턴' '포레스트' '폭력' '표절'
 '표현' '프로그래밍' '프로그램' '필요' '하위' '학술지' '학습'

,가능,각도,강화,개념,개선,개입,객체,거래,거리,검사,결정,결합,경쟁,경험,계면,계산,고정,과학,관계,관련,구조,국가,군집,규칙,극대,금융,기계,기반,기술,기하학,나무,내전,넷플릭스,농민,능력,다층,대량,대신,데이터,도구,도용,독일,동의어,라벨,랜덤,러닝,로지스틱,마이닝,마진,맞춤,...,주식,주요,주행,중요,지능,지도,지표,진단,집단,징후,차이,차이점,최대,최적,추천,추출,측정,치열,컴퓨터,코드,코딩,코사인,콘텐츠,탐지,트레이딩,트리,특정,특징,파이썬,판단,패턴,포레스트,폭력,표절,표현,프로그래밍,프로그램,필요,하위,학술지,학습,한국,해결,핵심,행동,향상,혼란,확률,활용,회귀
0,0.0,0.190441,0.0,0.0,0.0,0.0,0.190441,0.0,0.159605,0.159605,0.0,0.0,0.0,0.0,0.0,0.190441,0.0,0.000000,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.137726,0.190441,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.095166,0.000000,0.000000,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.120755,0.0,0.190441,0.0,0.0,0.190441,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.159605,0.0,0.159605,0.0,0.0,0.000000,0.0,0.190441,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.190441,0.0,0.0,0.0,0.159605,0.190441,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.159605,0.0,0.0,0.0,0.0,0.000000,0.0
1,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.180070,0.180070,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.214861,0.0,0.0,0.0,0.0,0.18007,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.107368,0.214861,0.214861,0.0,0.0,0.18007,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.214861,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.180070,0.0,0.0,0.214861,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.214861,0.000000,0.0,0.0,0.0,0.180070,0.000000,0.0,0.0,0.0,0.0,0.214861,0.120595,0.214861,0.0,0.000000,0.0,0.0,0.0,0.0,0.136239,0.0


# <b>유사도 측정
* 카운트 기반 임베딩 : TfidfVectorize
* 유사도 : cosine_similarity

In [10]:
tfidf_cos_matrix = cosine_similarity(tfidf_df , tfidf_df) #모든 문서끼리 서로 얼마나 비슷한지 계산

In [11]:
tfidf_cos_matrix.shape

(8, 8)

In [12]:
tfidf_cos_df = pd.DataFrame(tfidf_cos_matrix, index=df['content'] , columns=df['content'])
tfidf_cos_df.head(1)

content,"유사도(Similarity)는 두 데이터 객체(문서, 벡터 등)가 얼마나 비슷한지 정량적으로 수치화한 지표로, 주로 0에서 1 사이(또는 -1~1) 값으로 표현됩니다. 높을수록(1에 가까울수록) 두 데이터가 유사하며, KCI 문헌 유사도 검사 서비스나 꼬맨틀 등에서 데이터 분석, 표절 검사, 추천 시스템의 핵심 기술로 사용됩니다. \n\n1. 주요 유사도 측정 방식\n코사인 유사도 (Cosine Similarity): 두 벡터 간의 사잇각을 이용하여 유사도를 측정하는 방식입니다. 방향의 유사성을 중요하게 보며, 각도가 좁을수록(0도에 가까울수록) 유사도가 1에 가까워집니다.\n유클리드 거리 (Euclidean Distance): 두 점 사이의 기하학적 거리를 계산하여 유사도를 측정합니다. 거리가 작을수록 유사도가 높다고 판단하며, 비유사도를 측정하는 방식입니다.","데이터 과학/AI: 군집화(Clustering), 분류(Classification), 데이터 라벨링 및 AI 모델 학습/진단에 활용됩니다.\n문서 및 코드 유사도: KCI(한국학술지인용색인) 등을 활용해 문서 간 유사성을 측정하여 표절을 검사하거나, 파이썬 등 코드 도용을 방지하기 위한 도구로 활용됩니다 KCI 문헌 유사도 검사 서비스.\n비유사도(Dissimilarity): 데이터 간의 거리를 나타내며, 0에 가까울수록 유사도가 높고 비유사도가 낮다는 것을 의미합니다. \n",전쟁은 국가 또는 정치 집단 사이의 폭력이나 무력을 사용하는 상태 또는 행동을 말한다. 특별히 둘 이상의 국가 간에 어떤 목적을 두고 수행되는 싸움이다. 그러나 독일 농민전쟁 같은 내전도 전쟁이다. 치열한 경쟁이나 혼란을 전쟁에 비유하여 말하기도 한다.,전쟁은 국가 또는 정치 집단 사이의 폭력이나 무력을 사용하는 상태 또는 행동을 말한다. 특별히 둘 이상의 국가 간에 어떤 목적을 두고 수행되는 싸움이다. 그러나 독일 농민전쟁 같은 내전도 전쟁이다. 치열한 경쟁이나 혼란을 전쟁에 비유하여 말하기도 한다.,"머신 러닝(Machine Learning, 기계 학습)은 컴퓨터가 명시적으로 프로그래밍되지 않고도 데이터와 경험을 통해 스스로 패턴을 학습하고 예측 모델을 빌드하여 성능을 개선하는 인공지능(AI)의 하위 분야입니다. 인공지능이 인간의 지능을 모방한다면, 머신 러닝은 데이터를 기반으로 규칙을 스스로 찾아내 의사결정을 내리는 수단입니다. \n핵심 정의 및 특징\n스스로 학습: 데이터를 분석하여 패턴을 찾고, 학습량이 많아질수록 더 빠르고 정확해집니다.\n기반 기술: 데이터와 알고리즘을 활용하여 인간과 유사한 방식으로 문제를 해결합니다.\n데이터 주도: 사람이 규칙을 코딩하는 대신, 기계가 데이터를 통해 직접 규칙을 찾습니다.","머신 러닝의 주요 유형\n지도 학습 (Supervised Learning): 라벨링된(정답이 있는) 데이터를 통해 학습하여 입력값에 대한 예측값(분류/회귀)을 학습합니다.\n비지도 학습 (Unsupervised Learning): 라벨링되지 않은 데이터에서 숨겨진 구조나 패턴을 찾아냅니다.\n강화 학습 (Reinforcement Learning): 행동에 대한 보상을 통해 스스로 최적의 행동 방식을 학습합니다. \n\n머신 러닝 활용 예시\n추천 시스템: 유튜브, 넷플릭스 등의 맞춤 콘텐츠 추천.\n이미지 및 음성 인식: 사진 속 사물 분류, 음성 인식 서비스.\n사기 탐지 및 보안: 금융 거래의 이상 징후 감지.\n언어 모델: 챗봇 및 자동 번역 서비스.\n예측 모델: 주식 트레이딩 및 에너지 부하 예측. \n\n머신 러닝의 동의어 및 관련 용어\n기계 학습\nML (Machine Learning)\n데이터 마이닝 (부분집합)\n예측 분석 (Predictive Analytics) \n머신 러닝은 일반 프로그램이 고정된 규칙대로 움직이는 것과 달리, 데이터를 통해 경험을 쌓아 업무 수행 능력을 점진적으로 향상시키는 기술입니다.","머신러닝(기계학습)은 데이터에서 규칙을 찾아 학습하는 인공지능의 한 분야이며, 딥러닝은 이를 발전시킨 인공신경망 기반의 하위 개념입니다. 가장 큰 차이는 사람의 개입 여부로, 머신러닝은 사람이 특징을 정의해줘야 하지만, 딥러닝은 인공지능이 스스로 데이터의 특징을 추출하여 학습합니다. \n\n머신러닝 vs 딥러닝 주요 차이점\n구조: 머신러닝은 다양한 알고리즘(회귀, 의사결정 나무 등)을 사용하며, 딥러닝은 인간 뇌를 모방한 다층 신경망(Deep Neural Network)을 사용합니다.\n특징 추출 (Feature Extraction): 머신러닝은 사람이 데이터의 주요 특징을 추출해 주어야 하는 반면, 딥러닝은 구조화되지 않은 데이터(이미지, 음성 등)에서 스스로 특징을 자동 추출합니다.\n데이터 양: 머신러닝은 적은 데이터로도 학습 가능하지만, 딥러닝은 고성능 GPU와 대량의 데이터가 필요합니다.\n응용 분야: 머신러닝은 금융 데이터 분석, 스팸 메일 분류에 적합하며, 딥러닝은 자율주행, 얼굴 인식, AI 번역 등 복잡한 작업에 활용됩니다.","머신 러닝 알고리즘은 데이터를 분석하여 패턴을 학습하고, 예측이나 의사결정을 수행하는 수학적 방법론입니다. 주요 유형으로 정답이 있는 데이터를 학습하는 지도 학습(회귀, 분류), 정답 없이 구조를 찾는 비지도 학습(군집화), 행동에 따른 보상을 극대화하는 강화 학습으로 나뉩니다. \n\n주요 머신 러닝 알고리즘 분류 및 종류\n지도 학습 (Supervised Learning): 입력값과 목표값의 매핑 관계를 학습합니다.\n선형 회귀 (Linear Regression): 연속적인 수치(예: 주가, 매출)를 예측.\n로지스틱 회귀 (Logistic Regression): 데이터가 특정 범주에 속할 확률을 0~1 사이로 예측하여 분류.\n의사결정 트리 (Decision Tree): 스무고개 방식으로 데이터 분류 및 회귀 수행.\n랜덤 포레스트 (Random Forest): 여러 결정 트리를 결합하여 예측 성능을 높인 앙상블 알고리즘.\n서포트 벡터 머신 (SVM): 데이터 분류 경계면(마진)을 최대화하는 최적의 선을 찾음."
content,,,,,,,,
"유사도(Similarity)는 두 데이터 객체(문서, 벡터 등)가 얼마나 비슷한지 정량적으로 수치화한 지표로, 주로 0에서 1 사이(또는 -1~1) 값으로 표현됩니다. 높을수록(1에 가까울수록) 두 데이터가 유사하며, KCI 문헌 유사도 검사 서비스나 꼬맨틀 등에서 데이터 분석, 표절 검사, 추천 시스템의 핵심 기술로 사용됩니다. \n\n1. 주요 유사도 측정 방식\n코사인 유사도 (Cosine Similarity): 두 벡터 간의 사잇각을 이용하여 유사도를 측정하는 방식입니다. 방향의 유사성을 중요하게 보며, 각도가 좁을수록(0도에 가까울수록) 유사도가 1에 가까워집니다.\n유클리드 거리 (Euclidean Distance): 두 점 사이의 기하학적 거리를 계산하여 유사도를 측정합니다. 거리가 작을수록 유사도가 높다고 판단하며, 비유사도를 측정하는 방식입니다.",1.0,0.270651,0.063541,0.063541,0.097777,0.102904,0.041212,0.094443


In [13]:
search_text = "유사도"
search_list = tfidf_cos_df[tfidf_cos_df.index.str.contains(search_text)].index
print(f"검색 결과 : {len(search_list)}건\n")
for title_str in search_list[:5]:
    print(title_str)

검색 결과 : 2건

유사도(Similarity)는 두 데이터 객체(문서, 벡터 등)가 얼마나 비슷한지 정량적으로 수치화한 지표로, 주로 0에서 1 사이(또는 -1~1) 값으로 표현됩니다. 높을수록(1에 가까울수록) 두 데이터가 유사하며, KCI 문헌 유사도 검사 서비스나 꼬맨틀 등에서 데이터 분석, 표절 검사, 추천 시스템의 핵심 기술로 사용됩니다. 

1. 주요 유사도 측정 방식
코사인 유사도 (Cosine Similarity): 두 벡터 간의 사잇각을 이용하여 유사도를 측정하는 방식입니다. 방향의 유사성을 중요하게 보며, 각도가 좁을수록(0도에 가까울수록) 유사도가 1에 가까워집니다.
유클리드 거리 (Euclidean Distance): 두 점 사이의 기하학적 거리를 계산하여 유사도를 측정합니다. 거리가 작을수록 유사도가 높다고 판단하며, 비유사도를 측정하는 방식입니다.
데이터 과학/AI: 군집화(Clustering), 분류(Classification), 데이터 라벨링 및 AI 모델 학습/진단에 활용됩니다.
문서 및 코드 유사도: KCI(한국학술지인용색인) 등을 활용해 문서 간 유사성을 측정하여 표절을 검사하거나, 파이썬 등 코드 도용을 방지하기 위한 도구로 활용됩니다 KCI 문헌 유사도 검사 서비스.
비유사도(Dissimilarity): 데이터 간의 거리를 나타내며, 0에 가까울수록 유사도가 높고 비유사도가 낮다는 것을 의미합니다. 



In [18]:
search_title = "머신 러닝"
print(df[df['content'].str.contains(search_title, regex=False, na=False)]['content'] [:5] )

4                                                                                                                                                                                                                                           머신 러닝(Machine Learning, 기계 학습)은 컴퓨터가 명시적으로 프로그래밍되지 않고도 데이터와 경험을 통해 스스로 패턴을 학습하고 예측 모델을 빌드하여 성능을 개선하는 인공지능(AI)의 하위 분야입니다. 인공지능이 인간의 지능을 모방한다면, 머신 러닝은 데이터를 기반으로 규칙을 스스로 찾아내 의사결정을 내리는 수단입니다. \n핵심 정의 및 특징\n스스로 학습: 데이터를 분석하여 패턴을 찾고, 학습량이 많아질수록 더 빠르고 정확해집니다.\n기반 기술: 데이터와 알고리즘을 활용하여 인간과 유사한 방식으로 문제를 해결합니다.\n데이터 주도: 사람이 규칙을 코딩하는 대신, 기계가 데이터를 통해 직접 규칙을 찾습니다.
5    머신 러닝의 주요 유형\n지도 학습 (Supervised Learning): 라벨링된(정답이 있는) 데이터를 통해 학습하여 입력값에 대한 예측값(분류/회귀)을 학습합니다.\n비지도 학습 (Unsupervised Learning): 라벨링되지 않은 데이터에서 숨겨진 구조나 패턴을 찾아냅니다.\n강화 학습 (Reinforcement Learning): 행동에 대한 보상을 통해 스스로 최적의 행동 방식을 학습합니다. \n\n머신 러닝 활용 예시\n추천 시스템: 유튜브, 넷플릭스 등의 맞춤 콘텐츠 추천.\n이미지 및 음성 인식: 사진 속 사물 분류, 음성 인식 서비스.\n사기 탐지 및 보안: 금융 거래의 이상 징후 감지.\n언어 모델: 챗봇 및 자동 번역 서비스.\n예측 모델: 주식 트레이딩 및 에너지 부하 예측. \n\n머신